# 08 — The mechanism, the field's damage metric, and H1 on prompts nobody has seen

Three experiments, about an hour of GPU between them. Each closes a specific gap that notebook 07
left open, and each has its decision rule written above its code.

| § | what | why it exists | cost |
|---|---|---|---|
| **§2** | `gen_only`, the complement of `prompt_only` | H2's mechanism claim currently has **one arm**. Notebook 07 shows steering the prompt keeps the effect without the damage; it never showed that steering the generated positions *causes* the damage. Until this runs, "the damage comes from generation-time steering" is an inference from an absence | ~10 min |
| **§3** | cross-entropy on harmless data, per scheme | Arditi et al. measured plain activation addition costing **3×–52×** more CE loss than directional ablation and switched intervention type because of it. That is the field's damage metric, and this project has never reported it. If prompt-gating collapses the CE penalty, H2 speaks directly to the reason the field abandoned addition | ~10 min |
| **§4** | H1's damage axis, pre-registered, on 48 unseen prompts | The strongest H1 evidence — per-input masking halving breakage at matched suppression — was found on an axis chosen *after* seeing the data. This registers it first and tests it on prompts neither earlier run touched | ~30 min |

Setup cells §0 and §1 are **byte-identical to notebook 07's**, including the replication gate, so a
disagreement between the two notebooks can only come from the sections below.

**Run order.** §0 → §1 → §2 → §3 → §4 → §5. Set `ADASS_LOAD_MODEL=1` before §0.1.

## §0 Setup — identical to notebook 07

In [ ]:
# %% 0.0 BOOTSTRAP -- run this first, always. Identical locally and on Colab.
import os, subprocess, sys
from pathlib import Path

GITHUB_REPO = "YarinShitrit/adass"                 # from `git remote -v`
DRIVE_DIR   = "/content/drive/MyDrive/adass"       # fallback if you skip GitHub

IN_COLAB = "google.colab" in sys.modules


def _find_root(start):
    """Walk up looking for the repo: a pyproject.toml sitting next to the adass package."""
    p = Path(start).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").is_file() and (cand / "adass" / "core.py").is_file():
            return cand
    return None


def _early_env():
    """Read a .env BEFORE the package exists. Mirrors adass.env, which is the canonical copy.

    Duplicated here because on Colab this cell runs before the repo is cloned and before anything
    is pip-installed -- and the GitHub token needed to perform the clone has to come from
    somewhere. Which is also why the repo's own .env cannot be that somewhere: .env is gitignored,
    so a clone never contains one. Keep a filled-in .env on Drive; it survives runtimes.
    """
    for p in (Path.cwd() / ".env", Path("/content/drive/MyDrive/adass/.env"),
              Path("/content/drive/MyDrive/.env"), Path("/content/.env"), Path.home() / ".env"):
        if p.is_file():
            for line in p.read_text(encoding="utf-8").splitlines():
                line = line.strip().removeprefix("export ")
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, _, v = line.partition("=")
                v = v.strip().strip("\"'")
                if v and not os.environ.get(k.strip()):
                    os.environ[k.strip()] = v
            print(f"loaded .env from {p}")
            return p
    return None


def _secret(name, prompt):
    """Environment (incl. .env) -> Colab Secrets -> prompt. Nothing is stored in the notebook."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata          # browser Colab frontend only
        v = userdata.get(name)
        if v:
            os.environ[name] = v
            return v
    except Exception:
        pass
    import getpass
    v = getpass.getpass(prompt)
    if v:
        os.environ[name] = v
    return v


def _clone_or_update(repo, token, dest):
    """Clone if absent, pull if already there. NEVER let the token reach a traceback.

    Two failures this exists for, both hit on 30 August. A kernel restart leaves /content intact,
    so `git clone` into an existing checkout dies with exit 128 and a message nobody sees; and
    `check=True` raises CalledProcessError, whose `.args` carries the tokenised URL straight into
    the traceback Colab prints and then saves into the notebook file. A leaked PAT is a worse
    outcome than a failed clone, so the token never travels with the exception.
    """
    dest = Path(dest)
    if (dest / ".git").is_dir():
        cmd, what = ["git", "-C", str(dest), "pull", "--quiet"], "pull"
    elif dest.exists() and any(dest.iterdir()):
        raise RuntimeError(f"{dest} exists and is not a git checkout. Remove it, or point ROOT at "
                           "the repo by hand.")
    else:
        cmd, what = ["git", "clone", "--quiet",
                     f"https://{token}@github.com/{repo}.git", str(dest)], "clone"
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        err = (r.stderr or "").strip()
        if token:
            err = err.replace(token, "<token>")
        raise RuntimeError(f"git {what} failed (exit {r.returncode}): {err[:400]}")
    print(f"git {what} ok -> {dest}")
    return dest


_early_env()
ROOT = _find_root(Path.cwd())

if IN_COLAB and ROOT is None:
    if not os.environ.get("GH_TOKEN") and Path("/content/drive").exists() is False:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            _early_env()
        except Exception:
            pass
    token = _secret("GH_TOKEN", "GitHub PAT (read access to the repo): ")
    if token:
        ROOT = _clone_or_update(GITHUB_REPO, token, "/content/adass")
    else:
        ROOT = _find_root(DRIVE_DIR) or Path(DRIVE_DIR)

assert ROOT is not None, "repo not found -- set GITHUB_REPO, or put the repo at DRIVE_DIR"
os.chdir(ROOT)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
elif str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import adass
adass.load_env()
# Both sources are GATED: HF_TOKEN needs google/gemma-2-2b-it AND walledai/AdvBench accepted.
# NOTE: HF_HUB_OFFLINE is deliberately NOT set -- nothing is cached on a fresh runtime.
from huggingface_hub import get_token
if not get_token():
    adass.require("HF_TOKEN", "HuggingFace token (gemma-2-2b-it + AdvBench accepted): ")

import torch
print(adass.paths.describe())
print(adass.env.status())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none -- everything past §1 will be very slow")

In [ ]:
# %% 0.1 Run flags, the truncation point, and the prior run.
import json, math, itertools
from collections import Counter

OUT = "week6_mechanism.json"       # adass.save_results resolves bare names to data/results/

LOAD_MODEL = os.environ.get("ADASS_LOAD_MODEL", "0") == "1"
print(f"LOAD_MODEL={LOAD_MODEL}   (set ADASS_LOAD_MODEL=1 for everything past §1)")
RESULTS = {}

# save_results MERGES at the top level, so no later cell can delete a section it did not compute.
# §0.1 is the one deliberate truncation point, and it fires only on a full run: a CPU-only pass
# cannot regenerate the GPU sections, so rotating them away would destroy the only copy. That is
# not hypothetical -- it happened on 21 August, and again in a milder form on 25 August, which is
# why the `and LOAD_MODEL` is there.
_out = adass.results_path(OUT)
_prev = _out.with_suffix(".prev.json")
if _out.exists() and LOAD_MODEL:
    _out.replace(_prev)
    print(f"rotated {_out.name} -> {_prev.name}  (full run: starting clean)")
elif _out.exists():
    print(f"CPU-only run: MERGING into existing {_out.name}, not rotating.")

# The verdict cells re-run on CPU against whatever the last GPU run left behind, so every one of
# them reads through `stored()` rather than off a local variable that only exists mid-run.
# WHERE THE RESULTS SURVIVE. A Colab runtime takes its filesystem with it when it is recycled,
# and on 27 August it did exactly that to a completed run: every section had saved, every save had
# gone to /content/adass, and /content/adass no longer existed. Only the notebook's printed cell
# outputs were left -- the rates, but none of the generations or per-item instrument arrays.
#
# adass.save_results mirrors every write to ADASS_MIRROR when that is set. Point it at Drive and
# the copy outlives the runtime. This block sets it up automatically on Colab and says so loudly
# when it cannot, because a run that is not mirrored is a run you may have to pay for twice.
if IN_COLAB and not os.environ.get("ADASS_MIRROR"):
    _drive = Path("/content/drive/MyDrive")
    if not _drive.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as _e:
            print(f"could not mount Drive ({_e})")
    if _drive.exists():
        os.environ["ADASS_MIRROR"] = str(_drive / "adass_results")
        print(f"mirroring every save to {os.environ['ADASS_MIRROR']}")
    else:
        print("!! NO MIRROR: results live only on this runtime and die with it.")
        print("!! Set ADASS_MIRROR to a Drive path, or copy data/results/ out before disconnecting.")
elif os.environ.get("ADASS_MIRROR"):
    print(f"mirroring every save to {os.environ['ADASS_MIRROR']}")

PRIOR = {}
for _p in (_out, _prev):
    if _p.exists():
        PRIOR = json.load(open(_p))
        print(f"prior run loaded from {_p.name}: {list(PRIOR)}")
        break


def stored(key, default=None):
    """This run's value if this run computed it, else the previous run's."""
    return RESULTS.get(key, PRIOR.get(key, default))

In [ ]:
# %% 0.2 Module provenance. The stored judge output is only reusable if the prompts still hash
# to the version it was produced under -- see HANDOVER trap 2 for what silent drift cost here.
print("adass      ", adass.__version__, "from", Path(adass.__file__).parent)
print("judge hash ", adass.judge_prompt_hash())
_stored_hash = json.load(open(adass.artifact("steps123_results.json")))["step3"]["prompt_hash"]
print("stored hash", _stored_hash,
      "-- MATCH" if adass.judge_prompt_hash() == _stored_hash else "-- CHANGED: do not reload")
assert adass.judge_prompt_hash() == _stored_hash, (
    "judge prompts changed: every comparison in this notebook against a stored week-4 number "
    "would be measuring two different instruments. Bump the version deliberately or revert.")

In [ ]:
# %% 0.3 Environment, splits, vectors. float16 is a STOP condition: Gemma-2 emits broken text in
# fp16, and that failure is visually identical to the degeneration this project studies.
import torch, transformers

DEV, DT = adass.pick_device(), adass.pick_dtype(adass.pick_device())
print("device", DEV, "| dtype", DT)
assert DT is not torch.float16, "float16: STOP. See README, environment check."

CONFIG = json.load(open(adass.paths.config()))
LAYER  = CONFIG["best_layer"]              # 10, set from evidence on 24 August
R16    = CONFIG["r16"]                     # 0.5537 -- relative strength of the raw vector at L16
assert LAYER == 10, f"config says layer {LAYER}; this notebook is written for the layer-10 point"

SPL     = adass.make_splits(seed=CONFIG["seed"])   # train_n MUST stay at its default 128 --
PROMPTS = SPL["harmless_test"]                     # train_n=160 shifts harmless_test by 32 items
assert len(PROMPTS) == 48
DIRS = torch.load(adass.artifact("refusal_dirs.pt"))
V    = DIRS[LAYER + 1]                             # refusal_dirs is indexed [layer + 1]
print(f"{len(PROMPTS)} test prompts | layer {LAYER} | ||V|| {float(V.norm()):.3f} "
      f"| r16 {R16:.4f}")

MAXNEW  = 128          # 48 tokens cannot show apology-then-answer; week 3 §2 is why this is 128
GEN_BS  = 8
KL_BS   = 4            # teacher-forced logits are [B, T, 256k]; 4 keeps a T4 inside its memory
KL_REF_TOKENS = 48     # the fixed reference text, as in week 3 §5.1
KL_WINDOW = 8          # see §5: the shared window that makes position schemes comparable

RESULTS["env"] = dict(load_model=LOAD_MODEL, device=DEV, dtype=str(DT), torch=torch.__version__,
                      transformers=transformers.__version__, layer=LAYER, r16=R16,
                      max_new_tokens=MAXNEW, n_prompts=len(PROMPTS),
                      gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
                      bf16_supported=(torch.cuda.is_bf16_supported()
                                      if torch.cuda.is_available() else None))
print("env:", RESULTS["env"])
print(adass.save_results(RESULTS, OUT))

MODEL = TOK = TO_CHAT = None
if LOAD_MODEL:
    MODEL, TOK, DT, DEV = adass.load_model()
    TO_CHAT = adass.make_chat_fn(TOK)
    print("model loaded")

## §1 Instruments, the gate, and the two helpers

The same two axes and the same blocking gate as notebook 07: the mechanical detector for coherence,
the binary judge for answering, the unsteered negative control, and dense at rel 1.0 reproduced
against the week-4 cell. `week5_h1h3.json` is loaded as the comparison baseline — every position
number in §2 has to sit alongside the ones already measured there, and §1.3 asserts that the two
runs share an operating point before anything is compared across them.

In [ ]:
# %% 1.1 Fit the mechanical thresholds on the anchors, and define the two axes.
GENS = json.load(open(adass.artifact("week3_generations.json")))
FIT = adass.fit_coherence_thresholds(GENS["no-steer"], GENS["dense/all m=2"])
for feat, d in FIT.items():
    print(f"  {feat:12} thr={d['threshold']:8.3f}  bacc={d['balanced_acc']:.3f}  margin={d['margin']:+.3f}")


def mech_broken(texts):
    return [adass.classify_mechanical(t, FIT)["broken"] for t in texts]


def judge_answered(prompts, texts):
    """None when the model is not loaded, so every section still completes."""
    if not LOAD_MODEL:
        return None
    return [o["answered"] for o in
            adass.local_judge_binary(MODEL, TOK, TO_CHAT, list(zip(prompts, texts)))]


def score(texts, prompts, answered=None):
    """The row every table in this notebook reports. Wilson CIs on all three rates.

    CIs are stored as [lo, hi] -- wilson_ci returns (point, lo, hi) and the point estimate is
    already the neighbouring field. Indexing rather than unpacking that tuple is what cost a
    blocking control its meaning on 23 August (WORKLOG correction 14), so the slice is explicit.
    """
    br = mech_broken(texts)
    ans = judge_answered(prompts, texts) if answered is None else answered
    n = len(texts)
    row = dict(n=n, broken=sum(br) / n, broken_ci=list(adass.wilson_ci(sum(br), n))[1:],
               matcher=adass.refusal_rate(texts))
    if ans is not None:
        clean = [(not b) and (not a) for b, a in zip(br, ans)]
        row.update(suppressed=1 - sum(ans) / n,
                   suppressed_ci=list(adass.wilson_ci(n - sum(ans), n))[1:],
                   clean_refusal=sum(clean) / n,
                   clean_refusal_ci=list(adass.wilson_ci(sum(clean), n))[1:],
                   judge_answered=ans)
    row["mech_broken"] = br
    return row


def fmt(row, label=""):
    s = f"{label:30} broken {row['broken']:6.1%}"
    if "clean_refusal" in row:
        s += f" | suppressed {row['suppressed']:6.1%} | CLEAN {row['clean_refusal']:6.1%}"
    if "kl" in row:
        s += f" | KL {row['kl']:6.3f}"
    return s + f" | matcher {row['matcher']:6.1%}"


def disjoint(a, b):
    """Do two [lo, hi] intervals fail to overlap? The only evidence a cell is DECIDED at n=48."""
    return a[1] < b[0] or b[1] < a[0]

In [ ]:
# %% 1.2 The replication gate, plus the negative control and the KL reference text.
#
# Three things at once, all from the unsteered and the dense rel-1.0 conditions:
#   - REF_TEXTS  -- the fixed unsteered continuation every KL in this notebook is measured on;
#   - the negative control -- unsteered must be ~0% suppressed and ~0% broken, or the
#     instruments are wrong before any comparison starts;
#   - the gate -- dense at rel 1.0 must land where week 4 §7 left it.
if LOAD_MODEL:
    HN10 = adass.mean_hidden_norm(MODEL, TOK, TO_CHAT, PROMPTS, LAYER, device=DEV)
    print(f"mean ||h|| at layer {LAYER}: {HN10:.1f}  (week 4 measured 170.9)")

    REF_TEXTS = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0,
                               max_new_tokens=KL_REF_TOKENS, batch_size=GEN_BS,
                               device=DEV, dtype=DT)
    ns_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0, max_new_tokens=MAXNEW,
                             batch_size=GEN_BS, device=DEV, dtype=DT)
    ns = score(ns_gens, PROMPTS)
    print(fmt(ns, "no-steer (negative control)"))

    V_ref = adass.rel_norm_rows(V, R16 * 1.0, HN10)      # the operating point, norm-matched
    d10_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=LAYER, vector=V_ref, mult=1.0,
                              positions="all", max_new_tokens=MAXNEW, batch_size=GEN_BS,
                              device=DEV, dtype=DT)
    d10 = score(d10_gens, PROMPTS)
    d10.update(adass.strength_row(V_ref, 1.0, HN10), label="dense", rel_factor=1.0,
               method="dense", sparsity=0.0, positions="all", n_prompts=len(PROMPTS))
    print(fmt(d10, "dense/all rel 1.0"))

    w4 = json.load(open(adass.artifact("week4_layers.json")))["s7_relative_grid"]["cells"]["L10/rel1.0"]["row"]
    print(f"\nweek 4 §7 L10/rel1.0: broken {w4['broken']:.1%}  clean {w4['clean_refusal']:.1%}")

    # The bar is 10 points, not equality: the 24 August run reproduced every rate in the sweep to
    # within 4.2% while only 14-48% of individual generations matched token-for-token, because a
    # T4 has no bfloat16 and greedy decoding is deterministic given identical numerics and not
    # otherwise. Rates are dtype-invariant here; text is not.
    gate = dict(
        neg_control_ok=bool(ns["broken"] <= 0.05 and ns["suppressed"] <= 0.05),
        clean_delta=d10["clean_refusal"] - w4["clean_refusal"],
        broken_delta=d10["broken"] - w4["broken"],
        h_norm=HN10, h_norm_w4=CONFIG["h_norms"][str(LAYER)])
    gate["repro_ok"] = bool(abs(gate["clean_delta"]) <= 0.10 and abs(gate["broken_delta"]) <= 0.10)
    gate["pass_"] = bool(gate["neg_control_ok"] and gate["repro_ok"])
    print(f"\nnegative control {'PASS' if gate['neg_control_ok'] else 'FAIL'} | "
          f"replication delta clean {gate['clean_delta']:+.1%} broken {gate['broken_delta']:+.1%} "
          f"-> {'PASS' if gate['repro_ok'] else 'FAIL'}")
    RESULTS["s1_gate"] = dict(gate=gate, no_steer=ns, dense_rel1=d10)
    RESULTS["s1_ref_texts"] = REF_TEXTS
    print(adass.save_results(RESULTS, OUT))
    assert gate["pass_"], "BLOCKING: fix this before running anything below."
else:
    HN10 = CONFIG["h_norms"][str(LAYER)]
    REF_TEXTS = (PRIOR.get("s1_ref_texts") or None)
    ns = d10 = None
    print(f"deferred: needs ADASS_LOAD_MODEL=1. Using stored ||h|| = {HN10}")

In [ ]:
# %% 1.3 The helpers §2-§4 run on, and the cross-run consistency check.
PRIOR5 = json.load(open(adass.artifact("week5_h1h3.json")))
_e5 = PRIOR5["env"]
print(f"week5 run: layer {_e5['layer']}  r16 {_e5['r16']}  dtype {_e5['dtype']}  n {_e5['n_prompts']}")
assert _e5["layer"] == LAYER and abs(_e5["r16"] - R16) < 1e-9 and _e5["n_prompts"] == len(PROMPTS), (
    "week5 ran at a different operating point -- §2 cannot be compared against it")
REL_STAR = PRIOR5["s2_damage_onset"]["rel_star"]
print(f"REL_STAR from week5: {REL_STAR}")
if LOAD_MODEL:
    _d5 = PRIOR5["s4_h1"]["cells"]["dense/rel1.0"]["row"]
    print(f"dense/rel1.0  here {d10['clean_refusal']:.1%} clean | week5 {_d5['clean_refusal']:.1%}"
          + ("  -- identical" if abs(d10["clean_refusal"] - _d5["clean_refusal"]) < 1e-9
             else "  -- MOVED: investigate before reading §2"))


def run_config(label, base_vec, rel_factor, positions="all", prompts=None, extra=None):
    """Generate + score one configuration at an exactly matched relative strength."""
    prompts = PROMPTS if prompts is None else prompts
    rel = R16 * rel_factor
    vec = adass.rel_norm_rows(base_vec, rel, HN10)
    g = adass.generate(MODEL, TOK, TO_CHAT, prompts, layer=LAYER, vector=vec, mult=1.0,
                       positions=positions, max_new_tokens=MAXNEW, batch_size=GEN_BS,
                       device=DEV, dtype=DT)
    row = score(g, prompts)
    row.update(adass.strength_row(vec, 1.0, HN10), label=label, rel_factor=rel_factor,
               rel_target=rel, positions=str(positions), n_prompts=len(prompts))
    adass.empty_cache(DEV)
    if extra:
        row.update(extra)
    return row, g, vec


def prior_row(section, key):
    """A row measured in the week-5 run, so §2 does not re-pay for cells that already exist.

    Safe because both runs are the same greedy decode at the same operating point on the same
    prompts, and §1.2 plus the check above verify exactly that before any of them is read.
    """
    return PRIOR5[section]["cells"][key]["row"]

## §2 The missing arm — pre-registered

**What notebook 07 established.** At relative strength 2.0, `prompt-only` suppresses 97.9% of
answers with **0% broken**, where `all` positions suppresses 95.8% with **45.8% broken**, at an
identical perturbation norm. Read forward, that says the effect is set by perturbing the prompt and
the damage is caused by perturbing during generation.

**Why that is not yet demonstrated.** It is one arm of a dissociation. Removing generation-time
steering removed the damage — but nothing has shown that generation-time steering *on its own*
produces damage rather than simply doing nothing. Both stories fit the data so far:

- **the mechanism story** — perturbing a token the model is *producing* pushes it off-distribution
  and the text degenerates; perturbing the prompt only changes what it decides to say;
- **the interaction story** — the damage needs both, and steering generated positions alone is
  as inert as `prompt-last` was.

`gen_only` separates them, and `gen_first_k_np` (the first 4 generated positions, no prompt pass)
locates the effect within generation if there is one.

**The decision rule, fixed before the run.** At relative strength 2.0, against the `all` and
`prompt-only` rows already measured:

| outcome | reading |
|---|---|
| `gen_only` broken CI overlaps `all` (damage reproduced) **and** its suppression is below `prompt-only` with disjoint CIs | **DISSOCIATION.** Effect is set at the prompt, damage is caused during generation. H2's mechanism claim stands as written |
| `gen_only` broken ≤ 10% **and** suppression ≤ 30% | **INTERACTION.** Neither component alone does much; the damage needs steering across the whole sequence. The claim must be restated as being about *duration*, not about generated positions |
| `gen_only` suppression CI overlaps `prompt-only` | **REDUNDANT.** The two components carry the same effect and the mechanism claim is wrong |
| anything else | report the four cells as a table and make no mechanism claim |

`prompt-last` is the honest control on all of this: it is a single steered position inside the
prompt, and it suppresses only 27.1% at rel 2.0 — so "the prompt sets the effect" is already known
not to mean "any one prompt position will do".

In [ ]:
# %% 2.1 gen_only and gen_first_k_np, at both relative strengths.
NEW_POS = [("gen-only", "gen_only"), ("gen-first-4-np", ("gen_first_k_np", 4))]
REL_POINTS = [1.0, REL_STAR]

if LOAD_MODEL:
    mech = {}
    for f in REL_POINTS:
        for label, spec in NEW_POS:
            row, gens, vec = run_config(f"dense/{label}", V, f, positions=spec)
            mech[f"{label}/rel{f}"] = dict(row=row, gens=gens)
            print(fmt(row, f"{label:16s} rel x{f}"))
    RESULTS["s2_gen_only"] = dict(cells=mech, rel_points=REL_POINTS,
                                  specs=[[a, str(b)] for a, b in NEW_POS])
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 2.2 The four cells side by side, then the pre-registered verdict.
s2 = stored("s2_gen_only")
if s2:
    f = REL_STAR
    rows = {"all": prior_row("s5_positions", f"all/rel{f}"),
            "prompt-only": prior_row("s5_positions", f"prompt-only/rel{f}"),
            "prompt-last": prior_row("s5_positions", f"prompt-last/rel{f}"),
            "gen-only": s2["cells"][f"gen-only/rel{f}"]["row"],
            "gen-first-4-np": s2["cells"][f"gen-first-4-np/rel{f}"]["row"]}
    print(f"at relative strength {f}x -- steered positions vs what they do\n")
    print(f"{'positions':16s} {'suppressed':>11s} {'broken':>9s} {'clean':>8s} {'matcher':>9s}")
    for k, r in rows.items():
        print(f"{k:16s} {r['suppressed']:11.1%} {r['broken']:9.1%} {r['clean_refusal']:8.1%} "
              f"{r['matcher']:9.1%}")

    go, allp, po = rows["gen-only"], rows["all"], rows["prompt-only"]
    damage_reproduced = not disjoint(go["broken_ci"], allp["broken_ci"])
    effect_below_prompt = disjoint(go["suppressed_ci"], po["suppressed_ci"]) and \
        go["suppressed"] < po["suppressed"]
    inert = go["broken"] <= 0.10 and go["suppressed"] <= 0.30
    redundant = not disjoint(go["suppressed_ci"], po["suppressed_ci"])

    verdict = ("DISSOCIATION -- effect at the prompt, damage during generation"
               if (damage_reproduced and effect_below_prompt) else
               "INTERACTION -- the damage needs the whole sequence; restate the claim"
               if inert else
               "REDUNDANT -- both components carry the effect; the mechanism claim is wrong"
               if redundant else
               "UNCLASSIFIED -- report the table, make no mechanism claim")
    print(f"\ngen_only: broken {go['broken']:.1%} (all: {allp['broken']:.1%}) | "
          f"suppressed {go['suppressed']:.1%} (prompt-only: {po['suppressed']:.1%})")
    print(f"VERDICT: {verdict}")
    RESULTS["s2_verdict"] = dict(verdict=verdict, rel=f,
                                 damage_reproduced=bool(damage_reproduced),
                                 effect_below_prompt=bool(effect_below_prompt),
                                 inert=bool(inert), redundant=bool(redundant),
                                 table={k: {kk: r[kk] for kk in
                                            ("suppressed", "broken", "clean_refusal", "matcher")}
                                        for k, r in rows.items()})
    print(adass.save_results(RESULTS, OUT))

    print("\nRead them (HANDOVER trap 8) -- a mechanism claim resting on unread text is a guess:")
    for i in (0, 1, 2):
        print(f"\n--- gen-only rel {f}, prompt: {PROMPTS[i]}")
        print(s2["cells"][f"gen-only/rel{f}"]["gens"][i][:340])
else:
    print("deferred: §2.1 has not run")

## §3 The field's own damage metric

Arditi et al. (2024, Table 9) measured cross-entropy loss on harmless data under each intervention,
relative to the unsteered model: activation addition costs **+0.089** on Gemma 2B against directional
ablation's **+0.011**, and 3× to 52× across five models. That measurement is why the field moved to
ablation. This project has never reported it, so H2 has been arguing against a decision whose
evidence it never put on the same axis.

CE here is the mean per-token negative log-probability of the **unsteered continuation** under the
steered model, on the same 48 harmless prompts and the same fixed reference text every KL in this
project uses. Two columns, for the same reason §5 of notebook 07 has two: a full-length CE under a
`first-k` gate averages mostly unsteered positions and flatters the gate, so a window shared by every
scheme is reported beside it.

This section makes no claim on its own — it puts the position schemes on the axis the literature
already uses, so that H2 is legible to a reader who does not trust our judge.

In [ ]:
# %% 3.1 CE on harmless data: baseline, position schemes, mask schemes.
CE_WINDOW = 8

if LOAD_MODEL:
    CHAT = [TO_CHAT(p) for p in PROMPTS]
    REF_WIN = [TOK.decode(TOK(t, add_special_tokens=False).input_ids[:CE_WINDOW])
               for t in REF_TEXTS]

    def ce(vec, positions="all", window=None):
        conts = REF_TEXTS if window is None else REF_WIN
        lp = adass.continuation_logprob(MODEL, TOK, CHAT, conts, LAYER, vec, 1.0,
                                        positions, batch_size=KL_BS, device=DEV, dtype=DT)
        return -float(lp.mean())

    base_full = ce(None, "all")
    base_win = ce(None, "all", CE_WINDOW)
    print(f"unsteered CE: full {base_full:.4f} | first-{CE_WINDOW} {base_win:.4f}\n")

    CE_SPECS = [("dense/all", V, "all"), ("dense/prompt-only", V, "prompt_only"),
                ("dense/gen-only", V, "gen_only"), ("dense/first-4", V, ("gen_first_k", 4)),
                ("dense/prompt-last", V, "prompt_last")]
    ce_rows = {}
    print(f"{'config':22s} {'CE full':>9s} {'dCE':>8s} {'CE win':>9s} {'dCE win':>9s}")
    for label, base, pos in CE_SPECS:
        vec = adass.rel_norm_rows(base, R16 * REL_STAR, HN10)
        cf, cw = ce(vec, pos), ce(vec, pos, CE_WINDOW)
        ce_rows[label] = dict(ce_full=cf, d_full=cf - base_full,
                              ce_win=cw, d_win=cw - base_win, positions=str(pos))
        print(f"{label:22s} {cf:9.4f} {cf-base_full:+8.4f} {cw:9.4f} {cw-base_win:+9.4f}")
        adass.empty_cache(DEV)

    RESULTS["s3_ce"] = dict(rows=ce_rows, base_full=base_full, base_win=base_win,
                            window=CE_WINDOW, rel_factor=REL_STAR,
                            note="mean per-token NLL of the unsteered continuation under steering; "
                                 "comparable in KIND to Arditi Table 9, not in absolute value -- "
                                 "different model version, vector and strength convention")
    print(adass.save_results(RESULTS, OUT))

    d_all = ce_rows["dense/all"]["d_win"]
    d_po = ce_rows["dense/prompt-only"]["d_win"]
    if d_all > 0:
        print(f"\nOn the shared {CE_WINDOW}-token window, prompt-gating changes the CE penalty by a "
              f"factor of {d_po/d_all:.2f} ({d_po:+.4f} against {d_all:+.4f}).")
        print("Arditi Table 9's ratio between activation addition and ablation on Gemma 2B was 8x;"
              "\nthat is the comparison this row is here to make available, not to win.")
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

## §4 H1's damage axis — registered first this time, on prompts nothing has seen

**What happened.** In notebook 07, per-input `absproj` masking at 90% sparsity held dense's
suppression while breaking far less: at n=96, 18.8% against dense's 42.7% and static's 41.7%, with
both intervals disjoint from it. That is the AdaSS claim in its original form. It is also **not what
was pre-registered** — the primary axis in the damage regime was clean refusal, on which the same
comparison is undecided. A hypothesis confirmed on an axis chosen after seeing the data is worth
exactly one thing: a reason to run it again, declared first.

**The prompts.** `harmless_test` is `harmless[144 : 144 + test_n]`, so raising `test_n` extends the
set without moving a single earlier item. Both previous runs used the first 96. `test_n=144` yields
48 prompts that no run, no mask, no threshold and no selection step in this project has ever seen.
§4.1 asserts the first 96 are unchanged rather than trusting it.

**The rule, fixed before the run.** At relative strength `REL_STAR`, sparsity 0.90, n=48 fresh:

> **Primary axis: `broken`.** `absproj` beats a comparator when their Wilson intervals on `broken`
> are disjoint and absproj's is lower, **and** their suppression intervals overlap — the effect must
> be matched for a damage comparison to mean anything.
>
> - beats **both** dense and static → **H1-damage SUPPORTED**
> - beats **dense only** → **PARTIAL**: sparsification helps, but not more than the cheap static mask
> - beats **neither** → **NOT SUPPORTED**: the n=96 result does not survive on unseen prompts

**Power, stated in advance so it cannot be argued afterwards.** At n=48 a rate carries roughly ±14
points, and the effect being tested is 18.8% against ~42%. Those intervals separate only if the
fresh-prompt rates land near the earlier ones. If the verdict comes back undecided, that is a
**power** result and not evidence of absence — the honest follow-up is a pooled n=144 estimate,
reported as pooled and therefore selection-contaminated, plus a note that a decisive test needs a
larger fresh set than this project has prompts for.

In [ ]:
# %% 4.1 The fresh split, and the masks built on it.
if LOAD_MODEL:
    SPL144 = adass.make_splits(seed=CONFIG["seed"], test_n=144)
    P144 = SPL144["harmless_test"]
    assert P144[:48] == PROMPTS, "the extended split moved the original 48 -- STOP"
    SPL96 = adass.make_splits(seed=CONFIG["seed"], test_n=96)
    assert P144[:96] == SPL96["harmless_test"], "the extended split moved the week-5 96 -- STOP"
    FRESH = P144[96:]
    print(f"{len(FRESH)} prompts, none of them seen by any earlier run")
    print("first three:", *[f"  - {p}" for p in FRESH[:3]], sep="\n")

    MU = adass.last_token_hidden(MODEL, TOK, TO_CHAT, SPL["harmless_train"],
                                 device=DEV).mean(1)[LAYER + 1]
    H_F = adass.last_token_hidden(MODEL, TOK, TO_CHAT, FRESH, device=DEV)[LAYER + 1]
    GRAD_F = adass.grad_scores(MODEL, TOK, TO_CHAT, FRESH, V, LAYER,
                               device=DEV, dtype=DT, batch_size=4)

    def build_fresh(method, sparsity=0.90):
        if method == "static":
            return (V * adass.static_mask(V, sparsity)).unsqueeze(0).expand(len(FRESH), -1).contiguous()
        out = torch.empty(len(FRESH), V.numel())
        for i in range(len(FRESH)):
            if method == "absproj":
                m = adass.adaptive_absproj_mask(V, H_F[i], MU, sparsity)
            elif method == "signed":
                m = adass.adaptive_signed_mask(V, H_F[i], MU, sparsity)
            elif method == "grad":
                m = adass.topk_mask(GRAD_F[i], sparsity)
            else:
                raise ValueError(method)
            out[i] = V * m
        return out
    print("masks ready")
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 4.2 Three arms on the fresh prompts.
if LOAD_MODEL:
    fresh_cells = {}
    for label, method in [("dense", None), ("static-0.90", "static"), ("absproj-0.90", "absproj")]:
        base = V if method is None else build_fresh(method)
        row, gens, _ = run_config(label, base, REL_STAR, prompts=FRESH)
        fresh_cells[label] = dict(row=row, gens=gens)
        print(fmt(row, f"{label:14s} rel x{REL_STAR} (fresh n={len(FRESH)})"))
    RESULTS["s4_fresh"] = dict(cells=fresh_cells, rel_factor=REL_STAR, n=len(FRESH),
                               prompts_from="harmless_test[96:144]")
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 4.3 The pre-registered damage-axis verdict, and the pooled secondary.
s4 = stored("s4_fresh")
if s4:
    cells = {k: v["row"] for k, v in s4["cells"].items()}
    a = cells["absproj-0.90"]
    print(f"{'arm':14s} {'broken':>9s} {'CI':>16s} {'suppressed':>11s} {'clean':>8s}")
    for k, r in cells.items():
        ci = f"[{r['broken_ci'][0]:.2f},{r['broken_ci'][1]:.2f}]"
        print(f"{k:14s} {r['broken']:9.1%} {ci:>16s} {r['suppressed']:11.1%} {r['clean_refusal']:8.1%}")

    beats = {}
    for other in ("dense", "static-0.90"):
        b = cells[other]
        matched = not disjoint(a["suppressed_ci"], b["suppressed_ci"])
        lower = disjoint(a["broken_ci"], b["broken_ci"]) and a["broken"] < b["broken"]
        beats[other] = bool(matched and lower)
        print(f"  absproj vs {other:12s} effect matched={matched}  less damage (disjoint)={lower}"
              f"  -> {'BEATS' if beats[other] else 'not decided'}")

    verdict = ("H1-DAMAGE SUPPORTED" if all(beats.values()) else
               "PARTIAL -- beats dense, not static" if beats["dense"] else
               "NOT SUPPORTED on unseen prompts")
    print(f"\nVERDICT: {verdict}")

    # Secondary, and labelled as what it is: the fresh 48 pooled with the 96 the contender was
    # selected on. More power, less independence. Never quote it as the headline.
    p5 = PRIOR5.get("s7_confirmation", {}).get("cells")
    if p5:
        print("\nSECONDARY, pooled n=144 (selection-contaminated -- the 96 chose this contender):")
        for k in ("dense", "static-0.90", "absproj-0.90"):
            if k in p5 and k in cells:
                r96, r48 = p5[k]["row"], cells[k]
                nb = round(r96["broken"] * r96["n"]) + round(r48["broken"] * r48["n"])
                n = r96["n"] + r48["n"]
                pt, lo, hi = adass.wilson_ci(nb, n)
                print(f"  {k:14s} broken {pt:6.1%}  [{lo:.3f}, {hi:.3f}]  n={n}")
    RESULTS["s4_verdict"] = dict(verdict=verdict, beats=beats, rel=REL_STAR, n=s4["n"],
                                 axis="broken (pre-registered in this notebook's §4 header)")
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: §4.2 has not run")

## §5 What this notebook settled

In [ ]:
# %% 5.1 Summary.
print("=" * 76)
for key, name in [("s1_gate", "replication gate"), ("s2_verdict", "H2 mechanism"),
                  ("s3_ce", "CE on harmless data"), ("s4_verdict", "H1 damage axis, fresh")]:
    s = stored(key)
    if not s:
        print(f"{name:26s} not run")
    elif "verdict" in s:
        print(f"{name:26s} {s['verdict']}")
    elif key == "s1_gate":
        print(f"{name:26s} {'PASS' if s['gate']['pass_'] else 'FAIL'}")
    else:
        r = s["rows"]["dense/prompt-only"]["d_win"], s["rows"]["dense/all"]["d_win"]
        print(f"{name:26s} prompt-only {r[0]:+.4f} vs all {r[1]:+.4f} (windowed dCE)")
print("=" * 76)

RESULTS["meta"] = dict(notebook="08_mechanism", layer=LAYER, r16=R16, rel_star=REL_STAR,
                       max_new_tokens=MAXNEW,
                       registered="§2 dissociation rule and §4 damage-axis rule were written "
                                  "above their code before either was run")
print(adass.save_results(RESULTS, OUT))